# Multi-DataFrame Integration and Relational Data Analysis

In [2]:
import pandas as pd
import numpy as np

customers = pd.DataFrame({
    'customer_id': [101, 102, 103, 104, 105, 106],
    'customer_name': ['Aman', 'Riya', 'Kabir', 'Neha', 'Arjun', 'Meera'],
    'city': ['Delhi', 'Mumbai', 'Delhi', 'Pune', 'Mumbai', 'Jaipur']
})

orders = pd.DataFrame({
    'order_id': [1001, 1002, 1003, 1004, 1005, 1006, 1007],
    'customer_id': [101, 102, 101, 104, 106, 103, 102],
    'product_id': [1, 2, 3, 1, 4, 3, 2],
    'quantity': [2, 1, 3, 1, 2, 4, 2],
    'status': [
        'Completed', 'Cancelled', 'Completed',
        'Completed', 'Completed', 'Cancelled', 'Completed'
    ]
})

products = pd.DataFrame({
    'product_id': [1, 2, 3, 4, 5],
    'product_name': ['Laptop', 'Shirt', 'Book', 'Headphones', 'Phone'],
    'category': ['Electronics', 'Clothing', 'Books',
                 'Electronics', 'Electronics'],
    'unit_price': [50000, 1200, 400, 2500, 30000]
})

payments = pd.DataFrame({
    'order_id': [1001, 1002, 1003, 1005, 1006, 1008],
    'payment_method': ['Card', 'UPI', 'Card', 'Cash', 'UPI', 'Card'],
    'payment_amount': [100000, 1200, 1200, 5000, 1600, 2500]
})

customer_segments = pd.DataFrame({
    'customer_id': [101, 102, 103, 104, 105],
    'segment': ['Premium', 'Regular', 'Regular', 'Premium', 'New']
})

#### Q1. Three-table merge

Combine orders, customers, and products to create a single order-level DataFrame containing customer name, city, product name, category, quantity, and status.

In [9]:
customer_order=pd.merge(customers,orders,
             on='customer_id',
             how='inner')
data=pd.merge(customer_order,products,
             on='product_id',
             how='inner')[['customer_name','city','product_name','category','quantity','status']]
data

,customer_name,city,product_name,category,quantity,status
0,Aman,Delhi,Laptop,Electronics,2,Completed
1,Aman,Delhi,Book,Books,3,Completed
2,Riya,Mumbai,Shirt,Clothing,1,Cancelled
3,Riya,Mumbai,Shirt,Clothing,2,Completed
4,Kabir,Delhi,Book,Books,4,Cancelled
5,Neha,Pune,Laptop,Electronics,1,Completed
6,Meera,Jaipur,Headphones,Electronics,2,Completed


In [11]:
data = (
    customers
    .merge(orders, on='customer_id', how='inner')
    .merge(products, on='product_id', how='inner')
    [['customer_name', 'city', 'product_name',
      'category', 'quantity', 'status']]
)

display(data)

,customer_name,city,product_name,category,quantity,status
0,Aman,Delhi,Laptop,Electronics,2,Completed
1,Aman,Delhi,Book,Books,3,Completed
2,Riya,Mumbai,Shirt,Clothing,1,Cancelled
3,Riya,Mumbai,Shirt,Clothing,2,Completed
4,Kabir,Delhi,Book,Books,4,Cancelled
5,Neha,Pune,Laptop,Electronics,1,Completed
6,Meera,Jaipur,Headphones,Electronics,2,Completed


#### Q2. Calculate order value

Using the merged order-level DataFrame, calculate order_value as quantity × unit_price.

In [16]:
order_product=pd.merge(orders,products,
                      how='inner',
                      on='product_id')
order_product['order_value']=order_product['quantity']*order_product['unit_price']
order_product

,order_id,customer_id,product_id,quantity,status,product_name,category,unit_price,order_value
0,1001,101,1,2,Completed,Laptop,Electronics,50000,100000
1,1002,102,2,1,Cancelled,Shirt,Clothing,1200,1200
2,1003,101,3,3,Completed,Book,Books,400,1200
3,1004,104,1,1,Completed,Laptop,Electronics,50000,50000
4,1005,106,4,2,Completed,Headphones,Electronics,2500,5000
5,1006,103,3,4,Cancelled,Book,Books,400,1600
6,1007,102,2,2,Completed,Shirt,Clothing,1200,2400


#### Q3. Add payment information

Left-merge payments into the order-level DataFrame using order_id. Preserve orders without payment records.

In [20]:
OPP=pd.merge(order_product,payments,
            how='left',
            on='order_id')
OPP

,order_id,customer_id,product_id,quantity,status,product_name,category,unit_price,order_value,payment_method,payment_amount
0,1001,101,1,2,Completed,Laptop,Electronics,50000,100000,Card,100000.0
1,1002,102,2,1,Cancelled,Shirt,Clothing,1200,1200,UPI,1200.0
2,1003,101,3,3,Completed,Book,Books,400,1200,Card,1200.0
3,1004,104,1,1,Completed,Laptop,Electronics,50000,50000,NaN,NaN
4,1005,106,4,2,Completed,Headphones,Electronics,2500,5000,Cash,5000.0
5,1006,103,3,4,Cancelled,Book,Books,400,1600,UPI,1600.0
6,1007,102,2,2,Completed,Shirt,Clothing,1200,2400,NaN,NaN


#### Q4. Identify payment mismatches

Find orders where:
- The order exists but no payment record exists.
- A payment exists for an order ID absent from orders.
Use a merge indicator or an equivalent reliable approach.

In [31]:
#The order exists but no payment record exists.
mismatch=OPP[(OPP['payment_method'].isna()) | (OPP['payment_amount'].isna())]
mismatch

,order_id,customer_id,product_id,quantity,status,product_name,category,unit_price,order_value,payment_method,payment_amount
3,1004,104,1,1,Completed,Laptop,Electronics,50000,50000,NaN,NaN
6,1007,102,2,2,Completed,Shirt,Clothing,1200,2400,NaN,NaN


In [33]:
OPP

,order_id,customer_id,product_id,quantity,status,product_name,category,unit_price,order_value,payment_method,payment_amount
0,1001,101,1,2,Completed,Laptop,Electronics,50000,100000,Card,100000.0
1,1002,102,2,1,Cancelled,Shirt,Clothing,1200,1200,UPI,1200.0
2,1003,101,3,3,Completed,Book,Books,400,1200,Card,1200.0
3,1004,104,1,1,Completed,Laptop,Electronics,50000,50000,NaN,NaN
4,1005,106,4,2,Completed,Headphones,Electronics,2500,5000,Cash,5000.0
5,1006,103,3,4,Cancelled,Book,Books,400,1600,UPI,1600.0
6,1007,102,2,2,Completed,Shirt,Clothing,1200,2400,NaN,NaN


#### Q5. Add customer segments

Merge customer_segments into the combined DataFrame. Retain all orders and identify customers whose segment is missing.

In [36]:
data=pd.merge(OPP,customer_segments,
             how='inner',
             on='customer_id')
data

,order_id,customer_id,product_id,quantity,status,product_name,category,unit_price,order_value,payment_method,payment_amount,segment
0,1001,101,1,2,Completed,Laptop,Electronics,50000,100000,Card,100000.0,Premium
1,1002,102,2,1,Cancelled,Shirt,Clothing,1200,1200,UPI,1200.0,Regular
2,1003,101,3,3,Completed,Book,Books,400,1200,Card,1200.0,Premium
3,1004,104,1,1,Completed,Laptop,Electronics,50000,50000,NaN,NaN,Premium
4,1006,103,3,4,Cancelled,Book,Books,400,1600,UPI,1600.0,Regular
5,1007,102,2,2,Completed,Shirt,Clothing,1200,2400,NaN,NaN,Regular


#### Q6. Validate join relationships

Use validate= when joining orders with customers and products. Select appropriate validation options and explain why each is suitable.

In [43]:
data=customers.merge(orders,
                    on='customer_id',
                    how='inner',
                    validate='one_to_many').merge(
                    products,
                    on='product_id',
                    how='inner',
                    validate='many_to_one'
                    )
data

,customer_id,customer_name,city,order_id,product_id,quantity,status,product_name,category,unit_price
0,101,Aman,Delhi,1001,1,2,Completed,Laptop,Electronics,50000
1,101,Aman,Delhi,1003,3,3,Completed,Book,Books,400
2,102,Riya,Mumbai,1002,2,1,Cancelled,Shirt,Clothing,1200
3,102,Riya,Mumbai,1007,2,2,Completed,Shirt,Clothing,1200
4,103,Kabir,Delhi,1006,3,4,Cancelled,Book,Books,400
5,104,Neha,Pune,1004,1,1,Completed,Laptop,Electronics,50000
6,106,Meera,Jaipur,1005,4,2,Completed,Headphones,Electronics,2500


#### Q7. Index-based join

Set customer_id as the index of both customers and customer_segments. Use .join() to combine them, then reset the index.

In [56]:
customers=customers.set_index('customer_id')
customers

,customer_name,city
customer_id,,
101,Aman,Delhi
102,Riya,Mumbai
103,Kabir,Delhi
104,Neha,Pune
105,Arjun,Mumbai
106,Meera,Jaipur


In [58]:
customer_segments=customer_segments.set_index('customer_id')
customer_segments

,segment
customer_id,
101,Premium
102,Regular
103,Regular
104,Premium
105,New


In [68]:
result=customers.join(customer_segments, how='left').reset_index()

result

,customer_id,customer_name,city,segment
0,101,Aman,Delhi,Premium
1,102,Riya,Mumbai,Regular
2,103,Kabir,Delhi,Regular
3,104,Neha,Pune,Premium
4,105,Arjun,Mumbai,New
5,106,Meera,Jaipur,NaN


#### Q8. Compare merge and join

Produce equivalent results using merge() and index-based join(). Compare the column structure and row counts.

In [77]:
data=pd.merge(customers,orders,
             on='customer_id',
             how='inner')
data

,customer_id,customer_name,city,order_id,product_id,quantity,status
0,101,Aman,Delhi,1001,1,2,Completed
1,101,Aman,Delhi,1003,3,3,Completed
2,102,Riya,Mumbai,1002,2,1,Cancelled
3,102,Riya,Mumbai,1007,2,2,Completed
4,103,Kabir,Delhi,1006,3,4,Cancelled
5,104,Neha,Pune,1004,1,1,Completed
6,106,Meera,Jaipur,1005,4,2,Completed


In [75]:
data=customers.join(orders,
                   how='inner')
data

,customer_name,city,order_id,customer_id,product_id,quantity,status


#### Q9. Build a category sales report

Using completed orders only, calculate:
- category
- total_quantity
- total_revenue
- unique_customers
- average_order_value
Sort the report by total_revenue in descending order.

In [104]:
order_product=pd.merge(orders,products,
                      on='product_id',
                      how='inner',
                      validate='many_to_one')
completed_orders = order_product.loc[
    order_product['status'] == 'Completed'
]


completed_orders

,order_id,customer_id,product_id,quantity,status,product_name,category,unit_price
0,1001,101,1,2,Completed,Laptop,Electronics,50000
2,1003,101,3,3,Completed,Book,Books,400
3,1004,104,1,1,Completed,Laptop,Electronics,50000
4,1005,106,4,2,Completed,Headphones,Electronics,2500
6,1007,102,2,2,Completed,Shirt,Clothing,1200


In [110]:
completed_orders['order_value'] = (
    completed_orders['quantity'] * completed_orders['unit_price'])

category_report = (
    completed_orders
    .groupby('category')
    .agg(
        total_quantity=('quantity', 'sum'),
        total_revenue=('order_value', 'sum'),
        unique_customers=('customer_id', 'nunique'),
        average_order_value=('order_value', 'mean')
    )
    .reset_index()
    .sort_values('total_revenue', ascending=False)
)
category_report

C:\Users\kavit\AppData\Local\Temp\ipykernel_16828\4265124952.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  completed_orders['order_value'] = (


,category,total_quantity,total_revenue,unique_customers,average_order_value
2,Electronics,5,155000,3,51666.666667
1,Clothing,2,2400,1,2400.000000
0,Books,3,1200,1,1200.000000


#### Q10. Create a data-quality audit

Build a report that identifies:
- Orders without matching customers.
- Orders without matching products.
- Orders without payment records.
- Payments without matching orders.
- Missing customer segments.

In [113]:
# 1. Orders without matching customers
orders_customer = orders.merge(
    customers,
    on='customer_id',
    how='left',
    indicator=True
)

orders_without_customers = orders_customer.loc[
    orders_customer['_merge'] == 'left_only',
    orders.columns
]


# 2. Orders without matching products
orders_product = orders.merge(
    products,
    on='product_id',
    how='left',
    indicator=True
)

orders_without_products = orders_product.loc[
    orders_product['_merge'] == 'left_only',
    orders.columns
]


# 3. Orders without payment records
orders_payment = orders.merge(
    payments,
    on='order_id',
    how='left',
    indicator=True
)

orders_without_payments = orders_payment.loc[
    orders_payment['_merge'] == 'left_only',
    orders.columns
]


# 4. Payments without matching orders
payments_order = payments.merge(
    orders,
    on='order_id',
    how='left',
    indicator=True
)

payments_without_orders = payments_order.loc[
    payments_order['_merge'] == 'left_only',
    payments.columns
]


# 5. Customers without matching customer segments
customers_segments = customers.merge(
    customer_segments,
    on='customer_id',
    how='left',
    indicator=True
)

missing_customer_segments = customers_segments.loc[
    customers_segments['_merge'] == 'left_only',
    customers.columns
]


# Display audit results
print("Orders without matching customers:")
display(orders_without_customers)

print("Orders without matching products:")
display(orders_without_products)

print("Orders without payment records:")
display(orders_without_payments)

print("Payments without matching orders:")
display(payments_without_orders)

print("Customers without matching customer segments:")
display(missing_customer_segments)

Orders without matching customers:


,order_id,customer_id,product_id,quantity,status


Orders without matching products:


,order_id,customer_id,product_id,quantity,status


Orders without payment records:


,order_id,customer_id,product_id,quantity,status
3,1004,104,1,1,Completed
6,1007,102,2,2,Completed


Payments without matching orders:


,order_id,payment_method,payment_amount
5,1008,Card,2500


Customers without matching customer segments:


,customer_name,city
customer_id,,
106,Meera,Jaipur
